# Caching with real libraries

This notebook runs the real-library blocks of the lessons *LRU, TTL and hit-rate arithmetic*, *Exact and semantic cache* and *Build a semantic cache end to end*. No GPU and no API key are needed. The sentence model (all-MiniLM-L6-v2, about 91 MB) is downloaded from the Hugging Face hub on first use. The outputs saved in the cells come from a real run with the pinned versions.

In [ ]:
!pip install -q cachetools==8.0.0 sentence-transformers==6.3.0 fastapi==0.141.1 httpx==0.28.1
print('installed')

## 1. LRU and TTL with cachetools 8.0.0

In [ ]:
from cachetools import LRUCache, TTLCache

lru = LRUCache(maxsize=2)
for key in ["leave", "sso", "leave", "trash"]:
    lru[key] = "answer for " + key          # cachetools has no get-and-count: reading and writing are dict operations
print("LRU keys:", list(lru))

now = [0]
ttl = TTLCache(maxsize=10, ttl=60, timer=lambda: now[0])
ttl["leave"] = "24 days"
for t in (59, 60):
    now[0] = t
    print(f"t={t}s", ttl.get("leave"))

LRU keys: ['leave', 'trash']
t=59s 24 days
t=60s None


## 2. Real cosines of the eight question pairs (sentence-transformers 6.3.0)

In [ ]:
from sentence_transformers import SentenceTransformer, util

PAIRS = [('How many days of annual leave does a full-time employee get?', 'What is the annual leave allowance for full-time staff?', True), ('How many days of annual leave does a full-time employee get?', 'How many vacation days do full-time employees receive?', True), ('By when must carried-over leave days be used?', 'When do my carried-over leave days expire?', True), ('How long are deleted tickets kept in the trash?', 'How many days are deleted tickets kept in the trash before they are removed?', True), ('How many seats does the Starter plan include?', 'How many seats does the Team plan include?', False), ('How many sick days can I take without a medical certificate?', 'From which sick day is a medical certificate required?', False), ('How many days of annual leave does a full-time employee get?', 'How many days of sick leave does a full-time employee get?', False), ('Which plans have single sign-on?', 'How do I turn on single sign-on for my team?', False)]
model = SentenceTransformer("all-MiniLM-L6-v2")
print("dimensions:", model.get_sentence_embedding_dimension())
sims = []
print("pair  cosine  same intent")
for i, (stored, new, same) in enumerate(PAIRS, 1):
    va, vb = model.encode([stored, new])
    sims.append(round(float(util.cos_sim(va, vb)), 3))
    print(f"{i:4} {sims[-1]:7.3f}  {same}")
print("threshold  hits  false hits")
for t in (0.75, 0.80, 0.82, 0.85, 0.90):
    hits = sum(1 for s, p in zip(sims, PAIRS) if s >= t and p[2])
    false_hits = sum(1 for s, p in zip(sims, PAIRS) if s >= t and not p[2])
    print(f"{t:9.2f} {hits:5} {false_hits:11}")

dimensions: 384
pair  cosine  same intent
   1   0.774  True
   2   0.803  True
   3   0.826  True
   4   0.954  True
   5   0.765  False
   6   0.795  False
   7   0.813  False
   8   0.575  False
threshold  hits  false hits
     0.75     4           3
     0.80     3           1
     0.82     2           0
     0.85     1           0
     0.90     1           0


## 3. The semantic cache with real vectors

In [ ]:
import re

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")     # the first run downloads the model, later runs read it from the local cache
n = sum(p.numel() for p in model.parameters())
print(f"parameters: {n:,}  weights as 32-bit numbers: {n * 4 / 1e6:.1f} MB")
_vectors = {}

def embed(text):
    if text not in _vectors:
        _vectors[text] = model.encode(text, normalize_embeddings=True)       # 384 numbers, scaled to length 1
    return _vectors[text]

def cosine(a, b):
    return float(a @ b)                                                      # both have length 1, so the dot product is the cosine
print("numbers per vector:", len(embed("hello")))
import re

def normalise(question):
    """lower case, keep letters and digits, single spaces: 'How many Days?' and 'how many days' become the same key"""
    return " ".join(re.findall(r"[a-z0-9]+", question.lower()))

# STAND-IN corpus: one passage per fact of the Orbit Labs documents (a real pipeline searches the chunks of all 8 documents)
PASSAGES = {
    "leave":   ("leave-policy",    "Full-time employees receive 24 days of paid annual leave per calendar year."),
    "sick":    ("leave-policy",    "Employees may take up to 10 paid sick days per calendar year without a medical certificate."),
    "carry":   ("leave-policy",    "Up to 5 unused annual leave days may be carried over; carried-over days must be used by 31 March."),
    "starter": ("product-faq",     "The Starter plan includes up to 5 seats."),
    "team":    ("price-list",      "The Team plan includes up to 50 seats."),
    "trash":   ("product-faq",     "Deleted tickets stay in the trash for 30 days and are then removed for good."),
    "sso":     ("product-faq",     "Single sign-on with SAML is available on the Business plan and above."),
    "pwd":     ("security-policy", "Passwords must have at least 14 characters (the minimum password length)."),
}

class Pipeline:
    """STAND-IN for retrieve + prompt + model: the best passage by cosine, and a SCRIPTED answer that quotes it. It counts its calls."""
    def __init__(self):
        self.calls = 0
        self.passages = dict(PASSAGES)
        self.vectors = {key: embed(text) for key, (doc, text) in self.passages.items()}

    def edit(self, key, new_text):
        """a document changes: the passage is replaced and its vector is rebuilt"""
        self.passages[key] = (self.passages[key][0], new_text)
        self.vectors[key] = embed(new_text)

    def answer(self, question):
        self.calls += 1
        vec = embed(question)
        best = max(self.vectors, key=lambda key: cosine(vec, self.vectors[key]))
        doc, text = self.passages[best]
        return f"{text} [{doc}]"

GUARD = {"starter", "team", "business", "enterprise", "sick", "parental"}    # words that change the intent (a stand-in guard)

def guard(question):
    """the set of intent-changing words in a question; two questions may share an answer only when these sets are equal"""
    return frozenset(w for w in re.findall(r"[a-z0-9]+", question.lower()) if w in GUARD)

class TwoLayerCache:
    """Layer 1: exact normalised key. Layer 2: cosine >= threshold (skipped when threshold is None). Both layers are scoped by tenant and
    index version, entries expire after `ttl` seconds on the injected clock, and at most `capacity` entries are kept (LRU)."""
    def __init__(self, capacity, ttl, threshold, clock, use_version=True, use_guard=True):
        self.capacity, self.ttl, self.threshold, self.clock = capacity, ttl, threshold, clock
        self.use_version, self.use_guard = use_version, use_guard
        self.data = {}              # (tenant, version, normalised question) -> entry; dict order = least to most recently used
        self.evictions = 0

    def scope(self, tenant, version):
        return (tenant, version if self.use_version else 0)

    def alive(self, key):
        """True when the entry is still valid (and moves it to the most recent end); an expired entry is removed"""
        if self.clock() - self.data[key]["stored"] >= self.ttl:
            del self.data[key]
            return False
        self.data[key] = self.data.pop(key)
        return True

    def get(self, tenant, version, question):
        """returns (answer or None, layer) where layer is 'exact', 'semantic' or 'miss'"""
        scope = self.scope(tenant, version)
        key = scope + (normalise(question),)
        if key in self.data and self.alive(key):
            return self.data[key]["answer"], "exact"
        if self.threshold is not None:
            vec, g = embed(question), guard(question)
            best, best_sim = None, 0.0
            for k in list(self.data):
                if k[:2] != scope or not self.alive(k):
                    continue
                e = self.data[k]
                sim = cosine(vec, e["vec"])
                if sim > best_sim and (not self.use_guard or e["guard"] == g):
                    best, best_sim = k, sim
            if best is not None and best_sim >= self.threshold:
                return self.data[best]["answer"], "semantic"
        return None, "miss"

    def put(self, tenant, version, question, answer):
        key = self.scope(tenant, version) + (normalise(question),)
        self.data.pop(key, None)
        while len(self.data) >= self.capacity:
            del self.data[next(iter(self.data))]          # the first key is the least recently used
            self.evictions += 1
        self.data[key] = {"vec": embed(question), "guard": guard(question), "answer": answer, "stored": self.clock()}

A, Z = "acme", "zen"                      # two tenants, i.e. two customers of the same service
LOG = [                                   # (tenant, question, intent); intent is the label that says which fact the question asks for
    (A, "How many days of annual leave does a full-time employee get?", "leave"),
    (A, "how many DAYS of annual leave does a full-time employee get", "leave"),
    (A, "What is the annual leave allowance for full-time staff?", "leave"),
    (A, "How many sick days can I take without a medical certificate?", "sick"),
    (A, "How many seats does the Starter plan include?", "starter"),
    (A, "How many seats does the Team plan include?", "team"),
    (Z, "How many days of annual leave does a full-time employee get?", "leave"),
    (A, "How many vacation days do full-time employees receive?", "leave"),
    (A, "By when must carried-over leave days be used?", "carry"),
    (A, "How long are deleted tickets kept in the trash?", "trash"),
    (Z, "How many seats does the Starter plan include?", "starter"),
    (A, "When do my carried-over leave days expire?", "carry"),
    (A, "Which plans have single sign-on?", "sso"),
    (A, "How many days are deleted tickets kept in the trash before they are removed?", "trash"),
    (A, "How many characters must a password have?", "pwd"),
    (Z, "How long are deleted tickets kept in the trash?", "trash"),
    (A, "What is the minimum password length?", "pwd"),
    (A, "How many sick days am I allowed without a doctor note?", "sick"),
    (A, "How many days of sick leave can an employee take?", "sick"),
    (Z, "Which plans have single sign-on?", "sso"),
    (A, "How many seats does the Starter plan include?", "starter"),
    (A, "What is the seat limit of the Starter plan?", "starter"),
    (A, "How many seats does the Team plan include?", "team"),
    ("EDIT", "trash", "Deleted tickets stay in the trash for 45 days and are then removed for good."),      # the FAQ changes: 30 days becomes 45
    (A, "How long are deleted tickets kept in the trash?", "trash"),
    (A, "How many days are deleted tickets kept in the trash before they are removed?", "trash"),
    (Z, "How long are deleted tickets kept in the trash?", "trash"),
    (A, "Which plans support single sign-on?", "sso"),
    (A, "How many days of annual leave does a full-time employee get?", "leave"),
    (Z, "What is the annual leave allowance for full-time staff?", "leave"),
    (A, "By when must carried-over leave days be used?", "carry"),
    (A, "How many characters must a password have?", "pwd"),
    (Z, "How many sick days can I take without a medical certificate?", "sick"),
    (A, "How many vacation days do full-time employees receive?", "leave"),
    (A, "When do my carried-over leave days expire?", "carry"),
    (A, "How many seats does the Team plan include?", "team"),
    (Z, "How long are deleted tickets kept in the trash?", "trash"),
    (A, "What is the annual leave allowance for full-time staff?", "leave"),
    (A, "Which plans have single sign-on?", "sso"),
    (A, "What is the minimum password length?", "pwd"),
]

T_EXACT, T_EMBED, T_SCAN, T_PIPE = 1, 15, 2, 2400      # milliseconds per step, ASSUMED (a real service must measure them)
TICK = 20                                              # the fake clock advances 20 seconds per request

def replay(use_cache=True, threshold=None, capacity=20, ttl=500, **flags):
    pipe, now, version, seen = Pipeline(), [0], 1, {}
    cache = TwoLayerCache(capacity, ttl, threshold, lambda: now[0], **flags) if use_cache else None
    lookup = (T_EXACT if use_cache else 0) + (T_EMBED + T_SCAN if threshold is not None else 0)     # what a miss pays before the pipeline starts
    s = dict(requests=0, hits=0, false_hits=0, stale=0, ms=0)
    for item in LOG:
        if item[0] == "EDIT":                          # a document changes: the passage and the index version change
            pipe.edit(item[1], item[2])
            version += 1
            continue
        tenant, question, intent = item
        now[0] += TICK
        s["requests"] += 1
        doc, text = pipe.passages[intent]
        right = f"{text} [{doc}]"                      # what the pipeline answers for this intent right now
        seen.setdefault(intent, set()).add(right)      # every answer that was right at some time
        answer, layer = cache.get(tenant, version, question) if cache else (None, "miss")
        if answer is None:
            answer = pipe.answer(question)
            assert answer == right, (question, answer)     # the stand-in pipeline itself is always right, so any error below belongs to the cache
            s["ms"] += lookup + T_PIPE
            if cache:
                cache.put(tenant, version, question, answer)
            continue
        s["hits"] += 1
        s["ms"] += T_EXACT if layer == "exact" else T_EXACT + T_EMBED + T_SCAN
        if answer != right:
            s["stale" if answer in seen[intent] else "false_hits"] += 1
    s["calls"] = pipe.calls
    s["evictions"] = cache.evictions if cache else 0
    return s

PAIRS = [   # (question in the cache, new question, same intent?)
    ("How many days of annual leave does a full-time employee get?", "What is the annual leave allowance for full-time staff?", True),
    ("How many days of annual leave does a full-time employee get?", "How many vacation days do full-time employees receive?", True),
    ("How many characters must a password have?", "What is the minimum password length?", True),
    ("How many seats does the Starter plan include?", "What is the seat limit of the Starter plan?", True),
    ("How many sick days can I take without a medical certificate?", "How many sick days am I allowed without a doctor note?", True),
    ("How many seats does the Starter plan include?", "How many seats does the Team plan include?", False),
    ("How many days of annual leave does a full-time employee get?", "How many days of sick leave can an employee take?", False),
]

print("threshold | guard on: hits false | guard off: hits false")
good = []
for t in (0.5, 0.6, 0.7, 0.8, 0.9):
    on, off = replay(threshold=t), replay(threshold=t, use_guard=False)
    print(f"{t:9.1f} | {on['hits']:13} {on['false_hits']:5} | {off['hits']:14} {off['false_hits']:5}")
    if on["false_hits"] == 0:
        good.append((on["hits"], t))
best_hits = max(h for h, t in good)
chosen = max(t for h, t in good if h == best_hits)
s = replay(threshold=chosen)
print(f"chosen {chosen}: pipeline calls {s['calls']}, hit rate {s['hits'] / s['requests']:.2f}, false {s['false_hits']}, stale {s['stale']}, mean {s['ms'] / s['requests']:.0f} ms")
for old, new, same in PAIRS:
    print(f"{cosine(embed(old), embed(new)):5.2f} {str(same):5} {new[:50]}")

parameters: 22,713,216  weights as 32-bit numbers: 90.9 MB
numbers per vector: 384
threshold | guard on: hits false | guard off: hits false
      0.5 |            20     4 |             21     7
      0.6 |            18     0 |             19     3
      0.7 |            17     0 |             19     3
      0.8 |            15     0 |             15     0
      0.9 |             7     0 |              7     0
chosen 0.6: pipeline calls 21, hit rate 0.46, false 0, stale 0, mean 1309 ms
 0.77 True  What is the annual leave allowance for full-time s
 0.80 True  How many vacation days do full-time employees rece
 0.81 True  What is the minimum password length?
 0.84 True  What is the seat limit of the Starter plan?
 0.83 True  How many sick days am I allowed without a doctor n
 0.77 False How many seats does the Team plan include?
 0.72 False How many days of sick leave can an employee take?


## 4. A cached /ask endpoint, exercised in-process with FastAPI 0.141.1 and TestClient

In [ ]:
import re
from cachetools import TTLCache
from fastapi import FastAPI, Header, Response
from fastapi.testclient import TestClient

FACTS = {"leave": "24 days of paid annual leave per calendar year.", "trash": "Deleted tickets stay in the trash for 30 days."}   # STAND-IN pipeline data
state = {"version": 1, "calls": 0}                  # the index version and the number of pipeline calls
now = [0.0]                                         # a fake clock moved by hand, so every run prints the same
cache = TTLCache(maxsize=100, ttl=60, timer=lambda: now[0])
app = FastAPI()

def normalise(question):
    return " ".join(re.findall(r"[a-z0-9]+", question.lower()))

def pipeline(question):
    state["calls"] += 1
    return FACTS["trash"] if "deleted" in question.lower() else FACTS["leave"]

@app.get("/ask")
def ask(q: str, response: Response, x_tenant: str = Header()):
    key = (x_tenant, state["version"], normalise(q))
    answer = cache.get(key)
    response.headers["X-Cache"] = "hit" if answer is not None else "miss"
    if answer is None:
        answer = cache[key] = pipeline(q)
    return {"answer": answer}

@app.post("/admin/reindex")
def reindex():
    state["version"] += 1                           # the documents changed: keys of the old version can never match again
    return {"version": state["version"]}

client = TestClient(app)                            # calls the app in this process: no server, no port
print("TestClient comes from:", TestClient.__module__)

def show(label, r):
    print(f"{label:32} {r.status_code}  X-Cache: {r.headers.get('x-cache', '-'):4} pipeline calls: {state['calls']}")

Q = "How many days of annual leave do I get?"
show("acme, first ask", client.get("/ask", params={"q": Q}, headers={"X-Tenant": "acme"}))
show("acme, same question in capitals", client.get("/ask", params={"q": Q.upper()}, headers={"X-Tenant": "acme"}))
show("zen, same question", client.get("/ask", params={"q": Q}, headers={"X-Tenant": "zen"}))
show("acme, no tenant header", client.get("/ask", params={"q": Q}))
now[0] = 61
show("acme, 61 seconds later", client.get("/ask", params={"q": Q}, headers={"X-Tenant": "acme"}))
show("reindex", client.post("/admin/reindex"))
show("acme, after the reindex", client.get("/ask", params={"q": Q}, headers={"X-Tenant": "acme"}))

TestClient comes from: starlette.testclient
acme, first ask                  200  X-Cache: miss pipeline calls: 1
acme, same question in capitals  200  X-Cache: hit  pipeline calls: 1
zen, same question               200  X-Cache: miss pipeline calls: 2
acme, no tenant header           422  X-Cache: -    pipeline calls: 2
acme, 61 seconds later           200  X-Cache: miss pipeline calls: 3
reindex                          200  X-Cache: -    pipeline calls: 3
acme, after the reindex          200  X-Cache: miss pipeline calls: 4


## Try next

Change the capacity, the TTL and the threshold, and watch the numbers move. Never paste a secret into a notebook cell.